# ViSQA Quickstart

This notebook demonstrates the core ViSQA pipeline: text-queried video segmentation and grounding.

In [ ]:
# Install if needed
# !pip install -e ..[all]
# !python scripts/download_weights.py --models all

In [ ]:
from visqa import ViSQAPipeline
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from IPython.display import Video

In [ ]:
# Initialize pipeline
pipeline = ViSQAPipeline.from_pretrained('visqa-base')
print('Pipeline ready!')

In [ ]:
# Run on a video
VIDEO_PATH = 'examples/sample.mp4'  # Replace with your video
QUERIES = [
    'the person walking',
    'the red car',
]

result = pipeline.run(
    video_path=VIDEO_PATH,
    queries=QUERIES,
    output_dir='outputs/quickstart/',
)

print(f'Processed {len(result.results)} queries')

In [ ]:
# View results for first query
qr = result.results[0]
print(f"Query: '{qr.query}'")
print(f"Detected in {(qr.masks.sum(axis=(1,2)) > 0).sum()} / {len(qr.masks)} frames")

# Show first frame with detection
from visqa.utils.video_io import VideoReader
frames = VideoReader(VIDEO_PATH).read_all()

detected_frame_idx = np.where(qr.masks.sum(axis=(1,2)) > 0)[0]
if len(detected_frame_idx) > 0:
    fidx = detected_frame_idx[0]
    frame = frames[fidx]
    mask = qr.masks[fidx]
    box = qr.boxes[fidx]
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    axes[0].imshow(frame)
    axes[0].set_title('Original Frame')
    axes[0].axis('off')
    
    axes[1].imshow(frame)
    axes[1].imshow(mask, alpha=0.5, cmap='Reds')
    axes[1].set_title('Segmentation Mask')
    axes[1].axis('off')
    
    axes[2].imshow(frame)
    x1, y1, x2, y2 = box
    rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2, edgecolor='red', facecolor='none')
    axes[2].add_patch(rect)
    axes[2].set_title(f'Bounding Box (score={qr.scores[fidx]:.2f})')
    axes[2].axis('off')
    
    plt.suptitle(f"Query: '{qr.query}' | Frame {fidx}")
    plt.tight_layout()
    plt.show()

In [ ]:
# Watch the output video
if result.output_video_path:
    Video(result.output_video_path, width=800)

In [ ]:
# Plot confidence scores over time
fig, axes = plt.subplots(len(result.results), 1, figsize=(14, 3 * len(result.results)))
if len(result.results) == 1:
    axes = [axes]

for ax, qr in zip(axes, result.results):
    ax.plot(qr.scores, linewidth=1.5)
    ax.fill_between(range(len(qr.scores)), qr.scores, alpha=0.3)
    ax.set_xlabel('Frame')
    ax.set_ylabel('Confidence')
    ax.set_title(f"Query: '{qr.query}'")
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()